In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
print("sklearn:", sklearn.__version__)

sns.set_style("whitegrid")
%matplotlib inline

sklearn: 1.7.2


In [2]:
df = pd.read_csv("../data/AB_NYC_2019.csv")
print("Shape:", df.shape)
df.head(3)

Shape: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365


In [3]:
df_clean = df.copy()

# Drop unusable columns
df_clean = df_clean.drop(columns=["id", "host_id", "host_name", "name"])

# Drop listings with price = 0
df_clean = df_clean[df_clean["price"] > 0].reset_index(drop=True)

# Fill missing reviews_per_month with 0 (means no reviews yet)
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

# Parse last_review as datetime, then engineer "days since last review"
df_clean["last_review"] = pd.to_datetime(df_clean["last_review"])
# Reference date = latest date in the dataset
ref_date = df_clean["last_review"].max()
df_clean["days_since_last_review"] = (ref_date - df_clean["last_review"]).dt.days
# If never reviewed, mark as a large value (e.g., 9999)
df_clean["days_since_last_review"] = df_clean["days_since_last_review"].fillna(9999)

# Binary flag: has the listing ever been reviewed?
df_clean["has_reviews"] = (df_clean["number_of_reviews"] > 0).astype(int)

# Drop last_review now that we extracted info from it
df_clean = df_clean.drop(columns=["last_review"])

print("Shape after cleaning:", df_clean.shape)
print("\nNew columns added:")
print("  - days_since_last_review")
print("  - has_reviews")
print("\nSample of new features:")
print(df_clean[["days_since_last_review", "has_reviews", "number_of_reviews"]].head())

Shape after cleaning: (48884, 13)

New columns added:
  - days_since_last_review
  - has_reviews

Sample of new features:
   days_since_last_review  has_reviews  number_of_reviews
0                   262.0            1                  9
1                    48.0            1                 45
2                  9999.0            0                  0
3                     3.0            1                270
4                   231.0            1                  9


In [4]:
# Approx coordinates of Manhattan center (Times Square)
MANHATTAN_LAT = 40.7580
MANHATTAN_LON = -73.9855

# Haversine distance in km (simplified — good enough for NYC scale)
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df_clean["dist_to_center_km"] = haversine_km(
    df_clean["latitude"], df_clean["longitude"],
    MANHATTAN_LAT, MANHATTAN_LON
)

# Host type: single-listing vs multi-listing host
df_clean["is_multi_host"] = (df_clean["calculated_host_listings_count"] > 1).astype(int)

# Log-transform the target
df_clean["price_log"] = np.log1p(df_clean["price"])

print("New features added: dist_to_center_km, is_multi_host, price_log")
print("\nSample:")
print(df_clean[["dist_to_center_km", "is_multi_host", "price_log"]].head())
print("\nDist to center (km) stats:")
print(df_clean["dist_to_center_km"].describe().round(2))

New features added: dist_to_center_km, is_multi_host, price_log

Sample:
   dist_to_center_km  is_multi_host  price_log
0          12.337898              1   5.010635
1           0.508366              1   5.420535
2           6.757240              0   5.017280
3           8.387034              0   4.499810
4           5.701496              0   4.394449

Dist to center (km) stats:
count    48884.00
mean         7.11
std          4.44
min          0.07
25%          3.82
50%          6.39
75%          9.42
max         35.90
Name: dist_to_center_km, dtype: float64


In [5]:
# Cap outliers at 99th percentile
price_cap = df_clean["price"].quantile(0.99)
nights_cap = df_clean["minimum_nights"].quantile(0.99)
host_cap = df_clean["calculated_host_listings_count"].quantile(0.99)

df_clean["price"] = df_clean["price"].clip(upper=price_cap)
df_clean["minimum_nights"] = df_clean["minimum_nights"].clip(upper=nights_cap)
df_clean["calculated_host_listings_count"] = df_clean["calculated_host_listings_count"].clip(upper=host_cap)

# Recompute price_log after clipping
df_clean["price_log"] = np.log1p(df_clean["price"])

print(f"Clipping caps — price: ${price_cap:.0f}, nights: {nights_cap:.0f}, host_count: {host_cap:.0f}")
print("Shape:", df_clean.shape)
print("\nPrice stats after clipping:")
print(df_clean["price"].describe().round(2))

Clipping caps — price: $799, nights: 45, host_count: 232
Shape: (48884, 16)

Price stats after clipping:
count    48884.00
mean       143.99
std        121.93
min         10.00
25%         69.00
50%        106.00
75%        175.00
max        799.00
Name: price, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "neighbourhood_group", "neighbourhood", "latitude", "longitude",
    "room_type", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count",
    "availability_365", "days_since_last_review", "has_reviews",
    "dist_to_center_km", "is_multi_host"
]

X = df_clean[feature_cols]
y = df_clean["price_log"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Column groups for the pipeline
num_cols = [
    "latitude", "longitude", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count",
    "availability_365", "days_since_last_review", "dist_to_center_km"
]
cat_cols = ["neighbourhood_group", "room_type", "has_reviews", "is_multi_host"]
high_card_col = ["neighbourhood"]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("\nNumeric cols:", len(num_cols))
print("Categorical cols:", cat_cols)
print("High-cardinality col:", high_card_col)

Train: (39107, 14)  Test: (9777, 14)

Numeric cols: 9
Categorical cols: ['neighbourhood_group', 'room_type', 'has_reviews', 'is_multi_host']
High-cardinality col: ['neighbourhood']


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from category_encoders import TargetEncoder

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

high_card_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", TargetEncoder(smoothing=10.0))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
    ("high", high_card_pipe, high_card_col)
])

print("✅ Preprocessor built for 14 features")

✅ Preprocessor built for 14 features


In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=20, max_features="log2",
        min_samples_split=5, min_samples_leaf=2,
        n_jobs=-1, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        min_samples_split=5, min_samples_leaf=2,
        random_state=42
    ),
}

results = []
fitted = {}

for name, model in models.items():
    print(f"\n🔄 Training {name}...")
    t0 = time.time()
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    y_pred_log = pipe.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_real = np.expm1(y_test)

    results.append({
        "Model": name,
        "Test R²": round(r2_score(y_test, y_pred_log), 4),
        "MAE ($)": round(mean_absolute_error(y_real, y_pred), 2),
        "RMSE ($)": round(np.sqrt(mean_squared_error(y_real, y_pred)), 2),
        "Time (s)": round(time.time() - t0, 1),
    })
    fitted[name] = pipe
    print(f"   ✅ R² = {results[-1]['Test R²']}, MAE = ${results[-1]['MAE ($)']}, time = {results[-1]['Time (s)']}s")

results_df = pd.DataFrame(results).sort_values("Test R²", ascending=False)
results_df


🔄 Training Ridge...
   ✅ R² = 0.5609, MAE = $51.88, time = 0.1s

🔄 Training Random Forest...
   ✅ R² = 0.623, MAE = $47.78, time = 0.6s

🔄 Training Gradient Boosting...
   ✅ R² = 0.62, MAE = $48.5, time = 16.6s


,Model,Test R²,MAE ($),RMSE ($),Time (s)
1,Random Forest,0.6230,47.78,94.08,0.6
2,Gradient Boosting,0.6200,48.50,94.19,16.6
0,Ridge,0.5609,51.88,100.06,0.1


/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [9]:
from sklearn.model_selection import RandomizedSearchCV

gb_param_dist = {
    "model__n_estimators": [200, 300, 400],
    "model__learning_rate": [0.03, 0.05, 0.08],
    "model__max_depth": [4, 5, 6, 7],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__subsample": [0.8, 0.9, 1.0],
}

gb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

gb_search = RandomizedSearchCV(
    gb_pipe,
    param_distributions=gb_param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("🚀 Tuning Gradient Boosting (20 combos × 3-fold CV)...")
gb_search.fit(X_train, y_train)

print("\n✅ Best params:")
for k, v in gb_search.best_params_.items():
    print(f"   {k} = {v}")
print(f"\nBest CV R²: {gb_search.best_score_:.4f}")

🚀 Tuning Gradient Boosting (20 combos × 3-fold CV)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best params:
   model__subsample = 0.9
   model__n_estimators = 200
   model__min_samples_leaf = 8
   model__max_depth = 7
   model__learning_rate = 0.08

Best CV R²: 0.6348


In [10]:
rf_param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [15, 20, 25, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.5],
}

rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_jobs=-1, random_state=42))
])

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("🚀 Tuning Random Forest (20 combos × 3-fold CV)...")
rf_search.fit(X_train, y_train)

print("\n✅ Best params:")
for k, v in rf_search.best_params_.items():
    print(f"   {k} = {v}")
print(f"\nBest CV R²: {rf_search.best_score_:.4f}")

🚀 Tuning Random Forest (20 combos × 3-fold CV)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

✅ Best params:
   model__n_estimators = 300
   model__min_samples_split = 5
   model__min_samples_leaf = 1
   model__max_features = sqrt
   model__max_depth = 15

Best CV R²: 0.6311


In [11]:
def evaluate(name, search):
    pipe = search.best_estimator_
    y_pred_log = pipe.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_real = np.expm1(y_test)
    return {
        "Model": name,
        "Train R²": round(r2_score(y_train, pipe.predict(X_train)), 4),
        "Test R²":  round(r2_score(y_test, y_pred_log), 4),
        "MAE ($)":  round(mean_absolute_error(y_real, y_pred), 2),
        "RMSE ($)": round(np.sqrt(mean_squared_error(y_real, y_pred)), 2),
    }

tuned_results = pd.DataFrame([
    evaluate("Tuned Random Forest",     rf_search),
    evaluate("Tuned Gradient Boosting", gb_search),
]).sort_values("Test R²", ascending=False)

tuned_results

,Model,Train R²,Test R²,MAE ($),RMSE ($)
1,Tuned Gradient Boosting,0.7332,0.6258,47.94,93.33
0,Tuned Random Forest,0.7800,0.6229,47.83,94.20


In [12]:
import joblib, gzip, os

# Save the tuned Gradient Boosting pipeline
final_pipe = gb_search.best_estimator_

model_path = "../models/airbnb_price_pipeline_v2.pkl.gz"
with gzip.open(model_path, "wb", compresslevel=3) as f:
    joblib.dump(final_pipe, f)

size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"✅ Saved to {model_path}")
print(f"File size: {size_mb:.2f} MB")

# Sanity check
with gzip.open(model_path, "rb") as f:
    loaded = joblib.load(f)

sample = loaded.predict(X_test.iloc[:5])
print("\nFirst 5 predictions ($):", np.expm1(sample).round(2))
print("Actual ($):              ", np.expm1(y_test.iloc[:5]).values.round(2))

✅ Saved to ../models/airbnb_price_pipeline_v2.pkl.gz
File size: 0.77 MB

First 5 predictions ($): [ 98.56  88.16 100.04  52.52  93.88]
Actual ($):               [99. 90. 80. 60. 90.]


In [13]:
import json

# Get unique values from the training set (what the model was actually trained on)
neighbourhoods = sorted(df_clean["neighbourhood"].unique().tolist())
boroughs = sorted(df_clean["neighbourhood_group"].unique().tolist())
room_types = sorted(df_clean["room_type"].unique().tolist())

# Save as JSON so the app can load it
meta = {
    "neighbourhoods": neighbourhoods,
    "boroughs": boroughs,
    "room_types": room_types,
}

with open("../models/app_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Saved app_metadata.json")
print(f"   Boroughs: {len(boroughs)} -> {boroughs}")
print(f"   Room types: {len(room_types)} -> {room_types}")
print(f"   Neighbourhoods: {len(neighbourhoods)} (showing first 5)")
print(f"   {neighbourhoods[:5]}")

✅ Saved app_metadata.json
   Boroughs: 5 -> ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
   Room types: 3 -> ['Entire home/apt', 'Private room', 'Shared room']
   Neighbourhoods: 221 (showing first 5)
   ['Allerton', 'Arden Heights', 'Arrochar', 'Arverne', 'Astoria']


In [14]:
feature_meta = {
    "feature_cols": feature_cols,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "high_card_col": high_card_col,
}

with open("../models/feature_meta.json", "w") as f:
    json.dump(feature_meta, f, indent=2)

print("✅ Saved feature_meta.json")
print(f"Total features: {len(feature_cols)}")
for c in feature_cols:
    print("  -", c)

✅ Saved feature_meta.json
Total features: 14
  - neighbourhood_group
  - neighbourhood
  - latitude
  - longitude
  - room_type
  - minimum_nights
  - number_of_reviews
  - reviews_per_month
  - calculated_host_listings_count
  - availability_365
  - days_since_last_review
  - has_reviews
  - dist_to_center_km
  - is_multi_host
